# 02 — Baseline Multimodal (hasta 10 imágenes)

Fine-tuning de **Qwen2.5-VL-3B-Instruct** con LoRA para detección de datos de contacto en listings MeLi.

| Parámetro | Valor |
|-----------|-------|
| Modelo base | Qwen2.5-VL-3B-Instruct (4-bit) |
| LoRA rank / alpha | r=8 / α=16 |
| GPU objetivo | T4 (14.5 GB) |
| Imágenes / listing | hasta 10 |
| Métrica principal | F2-score |

**Prerequisito:** haber corrido `01_build_splits.ipynb` para tener `train.csv` y `val.csv` en Drive.

## 0 — Setup y dependencias

In [ ]:
# Setup: monta Drive, clona/actualiza repo, instala deps base
exec(open('/content/drive/MyDrive/contact-detection/scripts/colab_setup.py').read())

In [ ]:
# Instalar Unsloth (requiere restart del kernel la primera vez)
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git',
], check=True)
print('✅ Unsloth instalado')

In [ ]:
import os, gc, json, sys
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import yaml
from pathlib import Path
from collections import Counter

from unsloth import FastVisionModel
from transformers import AutoProcessor, Trainer, TrainingArguments

# Módulos del proyecto
REPO_DIR = '/content/meli-contact-detection'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.data.dataset   import csv_to_dataset
from src.data.collator  import build_multimodal_collator, smoke_test
from src.inference.predict import predict_one
from src.engine.decision   import decide

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
torch.cuda.empty_cache()

print('✅ imports OK')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('VRAM total:', f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

## 1 — Configuración

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/contact-detection'
SPLITS_DIR  = f'{DRIVE_ROOT}/data/splits'
IMGS_DIR    = f'{DRIVE_ROOT}/data/images'   # imágenes descargadas (persistentes)
OUTPUTS_DIR = f'{DRIVE_ROOT}/outputs'
CONFIG_PATH = f'{REPO_DIR}/configs/qwen25_3b.yaml'

# ── Config desde YAML ─────────────────────────────────────────────────────────
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

MODEL_NAME   = cfg['model']['name']
LOAD_4BIT    = cfg['model']['load_in_4bit']
MAX_IMAGES   = cfg['data']['max_images']        # 10
IMG_MAX_SIDE = cfg['data']['img_max_side']       # 512
PROMPT_CHARS = cfg['data']['prompt_max_chars']   # 2500
SEED         = cfg['data']['seed']               # 42

# ── Tamaño del run ─────────────────────────────────────────────────────────────
# Ajustá según el tiempo disponible en la sesión de Colab:
#   N_TRAIN=500  → ~25 min descarga + ~15 min training (T4)
#   N_TRAIN=2000 → ~100 min descarga + ~60 min training (T4)
#   N_TRAIN=None → dataset completo (66.9k) — solo si las imágenes ya están en Drive
N_TRAIN = 1000   # <── modificar según el sprint
N_VAL   = 300

print('Modelo  :', MODEL_NAME)
print('4-bit   :', LOAD_4BIT)
print('LoRA r  :', cfg['peft']['r'], '/ alpha:', cfg['peft']['lora_alpha'])
print('Épocas  :', cfg['training']['num_train_epochs'])
print('N_TRAIN :', N_TRAIN, '  N_VAL:', N_VAL)
print('IMG dir :', IMGS_DIR)

## 2 — Dataset: descarga de imágenes y construcción

In [ ]:
# Verificar que los splits existen
for split in ['train.csv', 'val.csv']:
    p = Path(SPLITS_DIR) / split
    assert p.exists(), f"❌ No encontrado: {p}. Corré 01_build_splits.ipynb primero."
    df = pd.read_csv(p)
    print(f"  {split}: {len(df):,} filas, {df['RESULT'].mean():.2%} DC")

In [ ]:
# Construir datasets (descarga imágenes si no están en Drive)
# Las imágenes se guardan en Drive → no se re-descargan en runs siguientes
print('Construyendo train dataset...')
ds_train = csv_to_dataset(
    csv_path        = f'{SPLITS_DIR}/train.csv',
    img_dir         = f'{IMGS_DIR}/train',
    max_images      = MAX_IMAGES,
    img_max_side    = IMG_MAX_SIDE,
    prompt_max_chars= PROMPT_CHARS,
    seed            = SEED,
    limit           = N_TRAIN,
)

print('\nConstruyendo val dataset...')
ds_val = csv_to_dataset(
    csv_path        = f'{SPLITS_DIR}/val.csv',
    img_dir         = f'{IMGS_DIR}/val',
    max_images      = MAX_IMAGES,
    img_max_side    = IMG_MAX_SIDE,
    prompt_max_chars= PROMPT_CHARS,
    seed            = SEED,
    limit           = N_VAL,
)

from datasets import DatasetDict
ds = DatasetDict({'train': ds_train, 'validation': ds_val})
print(f'\nDataset listo: train={len(ds["train"]):,}  val={len(ds["validation"]):,}')

In [ ]:
# Smoke check del dataset
ex = ds['train'][0]
print('Campos     :', list(ex.keys()))
print('item_id    :', ex['item_id'])
print('Imágenes   :', len(ex['image_path']))
print('Prompt     :', ex['prompt_text'][:300], '...')
print('Answer     :', ex['answer'])
print('Label      :', ex['label'])

# Distribución de clases
for split_name, split_ds in ds.items():
    labels = split_ds['label']
    print(f"\n{split_name}: {len(labels):,} ej.  DC={sum(labels):,} ({sum(labels)/len(labels):.2%})")

In [ ]:
# Distribución de cantidad de imágenes por listing
n_imgs_train = [len(ex['image_path']) for ex in ds['train']]
n_imgs_val   = [len(ex['image_path']) for ex in ds['validation']]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, counts, title in zip(axes, [n_imgs_train, n_imgs_val], ['train', 'val']):
    ax.hist(counts, bins=range(1, MAX_IMAGES + 2), edgecolor='white', color='#1565C0')
    ax.set_xlabel('Imágenes por listing')
    ax.set_ylabel('Cantidad de listings')
    ax.set_title(f'{title} — media: {np.mean(counts):.1f} imgs/listing')
    ax.set_xticks(range(1, MAX_IMAGES + 1))
plt.tight_layout()
plt.show()

## 3 — Modelo + LoRA

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit = LOAD_4BIT,
    device_map   = {'': 0},
    max_memory   = {0: '13GB'},
)

model = FastVisionModel.get_peft_model(
    model,
    r              = cfg['peft']['r'],
    lora_alpha     = cfg['peft']['lora_alpha'],
    lora_dropout   = cfg['peft']['lora_dropout'],
    bias           = cfg['peft']['bias'],
    target_modules = cfg['peft']['target_modules'],
)
model.gradient_checkpointing_enable()

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ modelo cargado')
print(f'   Parámetros entrenables: {trainable_params:,} / {total_params:,} ({trainable_params/total_params:.3%})')
print(f'   VRAM allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB')
print(f'   VRAM reserved : {torch.cuda.memory_reserved()/1024**3:.2f} GB')

## 4 — Collator multimodal (con label masking)

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)

collator = build_multimodal_collator(processor)

# Smoke test: verifica que el masking es correcto (label_frac << 1)
smoke_test(collator, ds['train'], n=2)

## 5 — Entrenamiento

In [ ]:
t_cfg = cfg['training']

# eval_steps / save_steps: cada ~10% del total de pasos
steps_per_epoch = len(ds['train']) // (t_cfg['per_device_train_batch_size'] *
                                        t_cfg['gradient_accumulation_steps'])
total_steps     = steps_per_epoch * t_cfg['num_train_epochs']
checkpoint_freq = max(10, total_steps // 10)

print(f'steps/epoch    : {steps_per_epoch}')
print(f'total steps    : {total_steps}')
print(f'checkpoint_freq: {checkpoint_freq}')

In [ ]:
RUN_NAME = f"qwen25vl_3b_baseline_{N_TRAIN}ej"
OUTPUT_DIR = f"{OUTPUTS_DIR}/{RUN_NAME}"

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    per_device_train_batch_size = t_cfg['per_device_train_batch_size'],
    per_device_eval_batch_size  = t_cfg['per_device_train_batch_size'],
    gradient_accumulation_steps = t_cfg['gradient_accumulation_steps'],
    num_train_epochs            = t_cfg['num_train_epochs'],
    learning_rate               = t_cfg['learning_rate'],
    fp16                        = t_cfg['fp16'],
    logging_steps               = t_cfg.get('logging_steps', 2),
    eval_strategy               = 'steps',
    eval_steps                  = checkpoint_freq,
    save_strategy               = 'steps',
    save_steps                  = checkpoint_freq,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    report_to                   = 'none',
    remove_unused_columns       = False,
)

trainer = Trainer(
    model          = model,
    args           = training_args,
    train_dataset  = ds['train'],
    eval_dataset   = ds['validation'],
    data_collator  = collator,
)

gc.collect()
torch.cuda.empty_cache()

print('Iniciando entrenamiento...')
trainer.train()
print('✅ Entrenamiento completado')

In [ ]:
# Curva de loss
log_history = trainer.state.log_history
train_logs = [(l['step'], l['loss'])       for l in log_history if 'loss'      in l]
eval_logs  = [(l['step'], l['eval_loss'])  for l in log_history if 'eval_loss' in l]

if train_logs:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(*zip(*train_logs), label='train loss', color='#1565C0')
    if eval_logs:
        ax.plot(*zip(*eval_logs), 'o--', label='eval loss', color='#C62828')
    ax.set_xlabel('Paso')
    ax.set_ylabel('Loss')
    ax.set_title('Curva de loss')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6 — Guardar modelo en Drive

In [ ]:
print(f'Guardando en {OUTPUT_DIR}...')
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)   # necesario para reproducir inferencia

# Guardar también la config usada
import shutil
shutil.copy(CONFIG_PATH, f'{OUTPUT_DIR}/qwen25_3b.yaml')

print('✅ Modelo guardado en Drive')
print('   Archivos:', [f.name for f in Path(OUTPUT_DIR).iterdir() if not f.is_dir()])

## 7 — Evaluación en validación

In [ ]:
# Número de ejemplos de val a evaluar (puede ser mayor que N_VAL si querés más)
N_EVAL = min(N_VAL, len(ds['validation']))

print(f'Corriendo inferencia sobre {N_EVAL} ejemplos de val...')

results = []
for i in range(N_EVAL):
    ex  = ds['validation'][i]
    raw, parsed = predict_one(ex, model=model, processor=processor, max_new_tokens=200)
    decision    = decide(ex['item_id'], parsed)

    y_true = ex['label']
    y_pred = 1 if decision.is_dc else 0

    results.append({
        'item_id':    ex['item_id'],
        'y_true':     y_true,
        'y_pred':     y_pred,
        'raw_label':  decision.raw_label,
        'confidence': decision.confidence,
        'raw_output': raw,
    })

results_df = pd.DataFrame(results)
parse_errors = (results_df['confidence'] == 'parse_error').sum()
print(f'Parse errors: {parse_errors} / {N_EVAL} ({parse_errors/N_EVAL:.1%})')

In [ ]:
def compute_metrics(y_true, y_pred, beta=2.0):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    b2 = beta ** 2
    fbeta = (1 + b2) * precision * recall / (b2 * precision + recall) if (b2 * precision + recall) > 0 else 0.0
    return dict(tp=tp, tn=tn, fp=fp, fn=fn,
                precision=precision, recall=recall, f2=fbeta)

m = compute_metrics(results_df['y_true'], results_df['y_pred'])

print('=' * 40)
print('  MÉTRICAS EN VALIDACIÓN')
print('=' * 40)
print(f'  N eval       : {N_EVAL}')
print(f'  Parse errors : {parse_errors}')
print()
print(f'  Confusion matrix:')
print(f'              Pred 0    Pred 1')
print(f'  Real 0  :  {m["tn"]:>6}    {m["fp"]:>6}   (TN / FP)')
print(f'  Real 1  :  {m["fn"]:>6}    {m["tp"]:>6}   (FN / TP)')
print()
print(f'  Precision : {m["precision"]:.4f}')
print(f'  Recall    : {m["recall"]:.4f}')
print(f'  F2-score  : {m["f2"]:.4f}   ← métrica principal')
print('=' * 40)

In [ ]:
# Visualización
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Confusion matrix heatmap
ax = axes[0]
cm = np.array([[m['tn'], m['fp']], [m['fn'], m['tp']]])
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Pred 0', 'Pred 1'])
ax.set_yticklabels(['Real 0', 'Real 1'])
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha='center', va='center', fontsize=14,
            color='white' if v > cm.max()/2 else 'black')
ax.set_title('Confusion Matrix')
plt.colorbar(im, ax=ax)

# Barras de métricas
ax2 = axes[1]
metrics_vals = [m['precision'], m['recall'], m['f2']]
bars = ax2.bar(['Precision', 'Recall', 'F2-score'], metrics_vals,
               color=['#42A5F5', '#66BB6A', '#FFA726'])
ax2.set_ylim(0, 1.1)
ax2.set_title('Métricas de clasificación')
for bar, v in zip(bars, metrics_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.3f}',
             ha='center', va='bottom', fontsize=11)
ax2.axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='target F2=0.8')
ax2.legend()

plt.tight_layout()
plt.show()

## 8 — Análisis de errores

In [ ]:
fp_df = results_df[(results_df['y_true'] == 0) & (results_df['y_pred'] == 1)]
fn_df = results_df[(results_df['y_true'] == 1) & (results_df['y_pred'] == 0)]

print(f'Falsos Positivos (FP): {len(fp_df)}  — modelo dice DC, no es DC')
print(f'Falsos Negativos (FN): {len(fn_df)}  — modelo dice no-DC, sí es DC  ← más crítico (F2)')

print('\n--- Falsos Negativos (primeros 5) ---')
for _, row in fn_df.head(5).iterrows():
    print(f"  item_id: {row['item_id']}  |  raw_label: {row['raw_label']}  |  confidence: {row['confidence']}")
    print(f"  salida: {row['raw_output'][:200]}")
    print()

In [ ]:
# Guardar resultados de evaluación en Drive
eval_path = f'{OUTPUT_DIR}/eval_results.csv'
results_df.to_csv(eval_path, index=False)

# Guardar métricas como JSON
metrics_path = f'{OUTPUT_DIR}/metrics.json'
with open(metrics_path, 'w') as f:
    json.dump({
        'n_eval':       N_EVAL,
        'n_train':      N_TRAIN,
        'parse_errors': int(parse_errors),
        **{k: round(float(v), 4) for k, v in m.items()},
    }, f, indent=2)

print(f'✅ eval_results.csv → {eval_path}')
print(f'✅ metrics.json     → {metrics_path}')

---
**Siguiente paso:** analizar los FN, ajustar el prompt o aumentar `N_TRAIN` para el siguiente sprint.